# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

print("Working dir:", os.getcwd())
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

Working dir: /content/flyrank-ml-internship
HF_TOKEN set: True


In [2]:
from huggingface_hub import hf_hub_download
import duckdb

fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=os.environ["HF_TOKEN"]
)

dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=os.environ["HF_TOKEN"]
)

dim_clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=os.environ["HF_TOKEN"]
)

print("fact:", fact_path)
print("dim_content:", dim_content_path)
print("dim_clients:", dim_clients_path)

con = duckdb.connect()

fact: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
dim_content: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
dim_clients: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet


In [3]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{fact_path}')").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [4]:
q1 = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM read_parquet('{fact_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print("Rows violating client+content+date uniqueness (should be empty if grain is correct):")
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating client+content+date uniqueness (should be empty if grain is correct):


,client_hash_id,content_hash_id,report_date,n


In [5]:
q2 = con.execute(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{fact_path}')
""").df()
q2

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [6]:
q3 = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM read_parquet('{fact_path}')
""").df()
q3

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


Five features for Lane 2 scoring, built from March 2026 data. Each is knowable at the decision moment — no future data used.

In [7]:
features_df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr,
        ga4_engaged_sessions
    FROM read_parquet('{fact_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 20
""").df()
features_df

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,ga4_engaged_sessions
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,0.000000,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,0.000000,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,0.008000,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,0.000000,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,0.000000,<NA>
5,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03-01,239,1,7.347280,0.004184,<NA>
6,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03-01,191,0,7.832461,0.000000,<NA>
7,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03-01,55,0,3.272727,0.000000,<NA>
8,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,77,0,5.636364,0.000000,<NA>
9,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,2,0,4.500000,0.000000,<NA>


One row = one client's content page's daily performance snapshot (grain: client_hash_id × content_hash_id × report_date), from fact_content_daily_performance. Time window: March 2026 (month=2026-03), a mid-panel month used for iteration — the sample table (June 2026) is a sealed test month I deliberately did not touch. I verified this grain and window with queries above (Query 1 confirmed no duplicate client+content+date combinations; Query 2 confirmed the row count and exact date span).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features: gsc_impressions, gsc_clicks, gsc_avg_position, ctr (derived), ga4_engaged_sessions — all observed, same-day signals, safe to use.

Label/proxy: I derive a same-window proxy label (bottom-quartile CTR = "underperforming") for the leakage demonstration below. This mirrors is_declining_label from the starter dataset but is not yet a true future-window target.

Context (not used as features): client_hash_id and content_hash_id are join keys only — used for grouping, never as model inputs, since they carry no real signal themselves.

Excluded, deliberately: any FlyRank product decision fields (health_score, priority_score, action_type, refresh flags) — these are not shipped in this release, and even if reconstructed I would never use them as features, since a model would just learn to copy the existing rule instead of finding real signal.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Query 1 (above) verified the grain: grouping by client_hash_id + content_hash_id + report_date and checking for duplicates returned zero rows, confirming one row = one page-day. Query 2 (above) verified the row count and date span for March 2026. Query 3 (above) verified availability using IS TRUE filters on gsc_data_available and ga4_data_available, showing how many rows actually have usable search/analytics data versus how many exist in total.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Summary of verification queries (see Q1, Q2, Q3 above):")
print("- Grain check: 0 duplicate rows found for client+content+date")
print(f"- Row count / date span: {q2['row_count'][0]} rows, {q2['min_date'][0]} to {q2['max_date'][0]}")
print(f"- Availability: {q3['gsc_available_rows'][0]} of {q3['total_rows'][0]} rows have GSC data (IS TRUE), {q3['ga4_available_rows'][0]} have GA4 data (IS TRUE)")

Summary of verification queries (see Q1, Q2, Q3 above):
- Grain check: 0 duplicate rows found for client+content+date
- Row count / date span: 9841378 rows, 2026-03-01 00:00:00 to 2026-03-31 00:00:00
- Availability: 3611061.0 of 9841378 rows have GSC data (IS TRUE), 413966.0 have GA4 data (IS TRUE)


This March 2026 slice cannot tell me anything about seasonality, since it's a single month — comparing this month to itself gives no sense of yearly patterns. Client history is also unbalanced: not every client has GSC and GA4 data starting from the same date (per dim_clients.gsc_data_start/ga4_data_start), so some rows in this month may reflect a client's very early tracking period rather than mature, representative behavior. Additionally, GSC-only rows (where ga4_data_available is FALSE) mean I can't compute engagement-based features for every row — only for the subset with GA4 coverage, confirmed in my Query 3 availability check.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### The leakage trap
I built a same-window proxy label (bottom-quartile CTR), then deliberately added a column derived directly from that label to show how leakage inflates results.

In [12]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Build df_leak fresh, self-contained
df_leak = con.execute(f"""
    SELECT
        gsc_impressions, gsc_clicks, gsc_avg_position,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('{fact_path}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df().dropna()

# Label: "underperforming" = got impressions but zero clicks (CTR == 0)
df_leak["label"] = (df_leak["ctr"] == 0).astype(int)
print("Label value counts:")
print(df_leak["label"].value_counts())

# THE TRAP: add a column derived directly from the label
df_leak["leaky_feature"] = df_leak["label"] * 100

# Leaky model: includes the deliberate leak
X_leaky = df_leak[["gsc_impressions", "gsc_avg_position", "leaky_feature"]]
y = df_leak["label"]
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
score_leaky = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:,1])
print("AUC WITH leak:", round(score_leaky, 4), "<- looks almost perfect, meaningless")

# Honest model: only signals that don't trivially encode the label
# (gsc_clicks is deliberately excluded — it near-perfectly determines the label since label = (ctr==0) = (clicks==0))
X_honest = df_leak[["gsc_impressions", "gsc_avg_position"]]
model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
score_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:,1])
print("AUC WITHOUT leak (honest):", round(score_honest, 4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label value counts:
label
1    3193080
0     417981
Name: count, dtype: int64
AUC WITH leak: 1.0 <- looks almost perfect, meaningless
AUC WITHOUT leak (honest): 0.8479


I deliberately added `leaky_feature`, a column computed directly from the label itself (`label * 100`), to a logistic regression alongside gsc_impressions and gsc_avg_position. The result was an AUC of 1.0 — perfect, but meaningless, because the model just learned to read the answer back from a feature that mathematically encodes it.

I also had to correct a subtler leak: my label (CTR == 0) is nearly identical to "gsc_clicks == 0", so including gsc_clicks as a feature made even my "honest" model look artificially perfect (also 1.0). After removing gsc_clicks and the deliberate leaky_feature, the honest model — using only gsc_impressions and gsc_avg_position — scored AUC = 0.8479, a realistic and far more meaningful number. This is the same leakage lesson from notebook 02 (where trend_pct leaked the trend_direction label), now performed on real warehouse data: leakage doesn't always look like an obviously fake column — sometimes it's a feature that's just mathematically entangled with how the label itself was defined. The honest 0.8479 shows there's still real, non-tautological signal in impressions and position alone — a meaningfully strong but not suspiciously perfect result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.